# Configurações adicionais de campo

Vimos várias configurações de modelo que se aplicam a todos os campos. Várias delas podem ser feitas (ou sobrescritas) **por campo**.

In [1]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

## Coerção strict × lax

In [2]:
class Model(BaseModel):
    model_config = ConfigDict(strict=True)

    field_1: bool = False
    field_2: bool = False

Assim, o modo strict vale para **todos** os campos do modelo.

In [3]:
try:
    Model(field_1=1.0, field_2=1.0)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
field_1
  Input should be a valid boolean [type=bool_type, input_value=1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/bool_type
field_2
  Input should be a valid boolean [type=bool_type, input_value=1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/bool_type


Em strict, o Pydantic não converte o float `1.0` em `True` (nem `0.0` em `False`).

Às vezes queremos a maioria dos campos em lax e só alguns em strict. Ou o contrário.

Então deixamos o modelo em lax (o padrão) e só alguns campos em strict:

In [4]:
class Model(BaseModel):
    model_config = ConfigDict(
        strict=False,
        validate_default=True,
    )

    field_1: bool = Field(strict=True, default=False)
    field_2: bool = False

In [5]:
print(f"Model(field_1=True, field_2=1.0): {Model(field_1=True, field_2=1.0)!r}")

Model(field_1=True, field_2=1.0): Model(field_1=True, field_2=True)


Mas isto não funciona:

In [6]:
try:
    Model(field_1=1.0, field_2=0.0)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field_1
  Input should be a valid boolean [type=bool_type, input_value=1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/bool_type


Ou o contrário: o modelo inteiro strict e alguns campos lax:

In [7]:
class Model(BaseModel):
    model_config = ConfigDict(
        strict=True,
        validate_default=True,
    )

    field_1: bool = Field(strict=False, default=False)
    field_2: bool = False

In [8]:
print(f"Model(field_1=1.0, field_2=True): {Model(field_1=1.0, field_2=True)!r}")

Model(field_1=1.0, field_2=True): Model(field_1=True, field_2=True)


Mas isto não funciona:

In [9]:
try:
    Model(field_1=1.0, field_2=1.0)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field_2
  Input should be a valid boolean [type=bool_type, input_value=1.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/bool_type


## Validar defaults

O mesmo vale para `validate_default`: podemos definir no modelo ou sobrescrever a configuração do modelo no campo.

In [10]:
class Model(BaseModel):
    model_config = ConfigDict(
        strict=False,
        validate_default=True,
    )

    field_1: bool = Field(strict=True, default=1.0, validate_default=False)
    field_2: bool = False

E agora, mesmo com `field_1` em strict, como pulamos explicitamente a validação do default de `field_1`, terminamos com dados inconsistentes:

In [11]:
print(f"Model(): {Model()!r}")

Model(): Model(field_1=1.0, field_2=False)


## Campos congelados

Vimos como congelar o modelo inteiro:

In [12]:
class Model(BaseModel):
    model_config = ConfigDict(frozen=True)

    field_1: int
    field_2: int

In [13]:
m = Model(field_1=1, field_2=2)

In [14]:
try:
    m.field_1=10
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field_1
  Instance is frozen [type=frozen_instance, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/frozen_instance


Alternativamente, podemos congelar só alguns campos.

Repare que não dá para congelar o modelo inteiro e "descongelar" campos individuais: com o modelo congelado, tudo fica congelado.

In [15]:
class Model(BaseModel):
    field_1: int = Field(frozen=True)
    field_2: int

In [16]:
m = Model(field_1=1, field_2=2)
print(f"m: {m!r}")

m: Model(field_1=1, field_2=2)


In [17]:
m.field_2 = 20
print(f"m: {m!r}")

m: Model(field_1=1, field_2=20)


In [18]:
try:
    m.field_1 = 10
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
field_1
  Field is frozen [type=frozen_field, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/frozen_field


## Excluindo campos da serialização

Vimos como passar argumentos a `model_dump()` e `model_dump_json()` para incluir ou excluir campos da serialização.

In [19]:
class Model(BaseModel):
    field_1: int = 1
    field_2: int = 2
    field_3: int = 3

In [20]:
m = Model()

In [21]:
print(f"m.model_dump(exclude=['field_1']): {m.model_dump(exclude=['field_1'])!r}")

m.model_dump(exclude=['field_1']): {'field_2': 2, 'field_3': 3}


Às vezes queremos **sempre** excluir um campo da serialização — talvez um valor secreto que o modelo precisa, mas que não queremos expor
por acidente na saída de um endpoint REST, por exemplo.

Em vez de depender de lembrar de excluí-lo a cada chamada de dump, definimos no campo:

In [22]:
class Model(BaseModel):
    key: str = Field(default='python', exclude=True)
    field_1: int = 1
    field_2: int = 2
    field_3: int = 3

In [23]:
m = Model()
print(f"m: {m!r}")

m: Model(key='python', field_1=1, field_2=2, field_3=3)


In [24]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'field_1': 1, 'field_2': 2, 'field_3': 3}


Uma vez excluído no nível do campo, não dá para "desexcluir" na serialização:

In [25]:
print(f"m.model_dump(include=['key', 'field_1', 'field_2', 'field_3']): {m.model_dump(include=['key', 'field_1', 'field_2', 'field_3'])!r}")

m.model_dump(include=['key', 'field_1', 'field_2', 'field_3']): {'field_1': 1, 'field_2': 2, 'field_3': 3}


`key` não foi serializado.

> 💡 **Complemento:** para segredos, combine `exclude=True` com o tipo `SecretStr` — ele também esconde o valor no `repr` e em logs
> (repare que `key='python'` apareceu no `repr` acima).

In [26]:
from pydantic import SecretStr

class Credentials(BaseModel):
    user: str
    password: SecretStr = Field(exclude=True)

c = Credentials(user="ana", password="s3nh4")
print(f"c, c.model_dump(), c.password.get_secret_value(): {c, c.model_dump(), c.password.get_secret_value()!r}")

c, c.model_dump(), c.password.get_secret_value(): (Credentials(user='ana', password=SecretStr('**********')), {'user': 'ana'}, 's3nh4')
